In [1]:
records_A = [
    {"id": 1, "name": "John Doe", "dob": "1990-01-01"},
    {"id": 2, "name": "Jane Smith", "dob": "1985-05-12"},
]

records_B = [
    {"id": "A", "name": "Jon Doe", "dob": "1990-01-01"},
    {"id": "B", "name": "Jane Smit", "dob": "1985-05-12"},
]

### Create Q-grams

In [2]:
def get_qgrams(text, q=2):
    text = text.lower().replace(" ", "")
    return [text[i:i+q] for i in range(len(text) - q + 1)]

In [3]:
get_qgrams("John")

['jo', 'oh', 'hn']

### Creating a bloom filter

In [3]:
import hashlib
import numpy as np

def hash_qgram(qgram, seed, size):
    return int(hashlib.md5((qgram + str(seed)).encode()).hexdigest(), 16) % size

In [4]:
def create_bloom_filter(record, size=100, num_hashes=3):
    bf = np.zeros(size, dtype=int)

    # Combine attributes
    combined = record["name"] + record["dob"]
    qgrams = get_qgrams(combined)

    for qg in qgrams:
        for i in range(num_hashes):
            index = hash_qgram(qg, i, size)
            bf[index] = 1

    return bf

### Encode dataset

In [5]:
bf_A = [create_bloom_filter(r) for r in records_A]
bf_B = [create_bloom_filter(r) for r in records_B]

### Compare bloom filters

In [6]:
def dice_similarity(bf1, bf2):
    intersection = np.sum(bf1 * bf2)
    return (2 * intersection) / (np.sum(bf1) + np.sum(bf2))

In [7]:
for i, a in enumerate(bf_A):
    for j, b in enumerate(bf_B):
        sim = dice_similarity(a, b)
        print(f"A{i} vs B{j}: {sim:.3f}")

A0 vs B0: 0.862
A0 vs B1: 0.353
A1 vs B0: 0.308
A1 vs B1: 0.933
